# Petit essai d'adaptation TTS Fon sur Google Colab

Ce notebook prépare un **pilote limité** du modèle `facebook/mms-tts-fon` avec les enregistrements autorisés du locuteur adulte 4. Il utilise un GPU Colab, conserve le paquet et le résultat dans ton Google Drive, et compare huit phrases de validation au modèle MMS original et aux enregistrements réels.

## Avant de lancer

1. Dans Colab, choisis **Exécution > Modifier le type d'exécution > GPU**.
2. Téléverse `speaker_4_colab_bundle.zip` dans `Mon Drive/fon_tts/`. Le paquet ne contient que les 610 paires d'entraînement, les 62 paires de validation et leurs huit références MMS.
3. Exécute les cellules dans l'ordre et autorise l'accès à Drive.

Les enregistrements et le checkpoint adapté restent privés dans ton Drive; ce notebook ne les publie pas sur Hugging Face. Le pilote ne prend que 64 exemples d'entraînement et 16 de validation : il vérifie que le pipeline fonctionne, mais ne suffit pas à conclure que la voix est améliorée. Le modèle MMS est sous licence CC BY-NC 4.0; vérifie cette licence avant toute utilisation ou diffusion du modèle adapté.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU CUDA détecté. Dans Colab, sélectionne Exécution > Modifier le type d'exécution > GPU, puis relance.")
print(torch.cuda.get_device_name(0))
print(f"CUDA disponible : {torch.version.cuda}")

In [ ]:
from pathlib import Path
import zipfile
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/fon_tts")
BUNDLE_PATH = DRIVE_ROOT / "speaker_4_colab_bundle.zip"
BUNDLE_ROOT = Path("/content/fon_speaker4_bundle")
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Dépose le paquet dans Mon Drive/fon_tts/ : {BUNDLE_PATH}")
BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as bundle:
    root = BUNDLE_ROOT.resolve()
    for member in bundle.infolist():
        target = (BUNDLE_ROOT / member.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError(f"Chemin inattendu dans l'archive : {member.filename}")
    bundle.extractall(BUNDLE_ROOT)
print(f"Paquet extrait dans {BUNDLE_ROOT}")

In [ ]:
%pip install -q "transformers==4.38.2" "datasets[audio]==3.6.0" "accelerate==0.27.2" "pyarrow==20.0.0" Cython

In [ ]:
import csv
import datasets
import json
import numpy as np
import unicodedata
import wave
from packaging.version import Version
from datasets import Audio, Dataset, DatasetDict
from transformers import AutoTokenizer

if Version(datasets.__version__) < Version("3.6.0"):
    raise RuntimeError(
        f"datasets 3.6.0 ou plus est requis pour NumPy 2; version active : {datasets.__version__}. "
        "Redémarre le runtime puis réexécute les cellules depuis l'installation."
    )
print(f"Versions vérifiées : datasets {datasets.__version__}, NumPy {np.__version__}")

def transcript_key(text):
    return " ".join(unicodedata.normalize("NFC", text).casefold().split())

def load_and_check_split(name):
    with (BUNDLE_ROOT / f"{name}.csv").open(encoding="utf-8", newline="") as stream:
        rows = list(csv.DictReader(stream))
    if not rows:
        raise ValueError(f"Manifeste {name} vide")
    audio_paths = []
    for row in rows:
        audio_path = (BUNDLE_ROOT / row["audio_path"]).resolve(strict=True)
        if BUNDLE_ROOT.resolve() not in audio_path.parents:
            raise ValueError(f"Audio hors du paquet : {audio_path}")
        with wave.open(str(audio_path), "rb") as wav:
            if (wav.getframerate(), wav.getnchannels(), wav.getsampwidth()) != (16000, 1, 2):
                raise ValueError(f"Format WAV inattendu : {audio_path.name}")
        audio_paths.append(str(audio_path))
        if not row["text"].strip():
            raise ValueError(f"Transcription vide : {audio_path.name}")
    return rows, audio_paths

train_rows, train_audio = load_and_check_split("train")
valid_rows, valid_audio = load_and_check_split("validation")
overlap = {transcript_key(row["text"]) for row in train_rows} & {
    transcript_key(row["text"]) for row in valid_rows
}
if overlap:
    raise ValueError(f"Fuite de {len(overlap)} transcription(s) entre les splits")

tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-fon")
unknown_rows = sum(
    tokenizer.unk_token_id in tokenizer(row["text"]).input_ids
    for row in train_rows + valid_rows
)
if unknown_rows:
    raise ValueError(f"{unknown_rows} transcription(s) contiennent un token inconnu")

def to_dataset(rows, paths):
    dataset = Dataset.from_dict({"audio": paths, "text": [row["text"] for row in rows]})
    return dataset.cast_column("audio", Audio(sampling_rate=16000))

DATASET_PATH = Path("/content/fon_dataset")
DatasetDict({
    "train": to_dataset(train_rows, train_audio),
    "eval": to_dataset(valid_rows, valid_audio),
}).save_to_disk(str(DATASET_PATH))
comparison = json.loads((BUNDLE_ROOT / "comparison/comparison.json").read_text(encoding="utf-8"))
if len(comparison) != 8:
    raise ValueError(f"8 phrases de comparaison attendues, trouvé : {len(comparison)}")
print(f"Données vérifiées : {len(train_rows)} train, {len(valid_rows)} validation, aucun texte partagé, aucun token inconnu.")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

COMMIT = "6f3f51f4d667f5c3eef89484d151ffd39d2c2b89"
REPO_DIR = Path("/content/finetune-hf-vits")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/ylacombe/finetune-hf-vits.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", COMMIT], cwd=REPO_DIR, check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if actual_commit != COMMIT:
    raise RuntimeError(f"Commit du fine-tuner inattendu : {actual_commit}")

# Le dépôt exige la compilation Cython de son extension monotonic_align avant l'entraînement.
ALIGNMENT_DIR = REPO_DIR / "monotonic_align"
(ALIGNMENT_DIR / "monotonic_align").mkdir(exist_ok=True)
subprocess.run(
    [sys.executable, "setup.py", "build_ext", "--inplace"],
    cwd=ALIGNMENT_DIR,
    check=True,
)
subprocess.run(
    [sys.executable, "-c", "from monotonic_align import maximum_path; print('monotonic_align compilé et importable')"],
    cwd=REPO_DIR,
    check=True,
)

# Matplotlib récent a supprimé tostring_rgb(); lire le buffer RGBA reste compatible.
plot_script = REPO_DIR / "utils" / "plot.py"
plot_source = plot_script.read_text(encoding="utf-8")
old_rgb_call = "fig.canvas.tostring_rgb()"
new_rgb_call = "np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()"
if plot_source.count(old_rgb_call) == 2:
    plot_script.write_text(plot_source.replace(old_rgb_call, new_rgb_call), encoding="utf-8")
elif plot_source.count(old_rgb_call) == 0 and plot_source.count(new_rgb_call) == 2:
    print("Compatibilité Matplotlib déjà configurée dans ce runtime.")
else:
    raise RuntimeError(
        "utils/plot.py inattendu : nombre d'appels tostring_rgb="
        f"{plot_source.count(old_rgb_call)}, lecture buffer_rgba={plot_source.count(new_rgb_call)}"
    )
print("Compatibilité des graphiques Matplotlib vérifiée.")

# Le script amont charge les jeux depuis le Hub; ce pilote remplace seulement ce bloc
# par le DatasetDict local préparé ci-dessus, sans téléverser les voix.
trainer_script = REPO_DIR / "run_vits_finetuning.py"
source = trainer_script.read_text(encoding="utf-8")
start_marker = "    raw_datasets = DatasetDict()\n"
end_marker = "    if data_args.audio_column_name not in next(iter(raw_datasets.values())).column_names:"
replacement = (
    '    raw_datasets = datasets.load_from_disk(os.environ["FON_DATASET_PATH"])\n'
    '    if not isinstance(raw_datasets, datasets.DatasetDict):\n'
    '        raise TypeError("Le jeu local doit être un DatasetDict")\n'
    '    if training_args.do_train and "train" not in raw_datasets:\n'
    '        raise ValueError("Split train manquant")\n'
    '    if training_args.do_eval and "eval" not in raw_datasets:\n'
    '        raise ValueError("Split eval manquant")\n\n'
)
local_dataset_marker = '    raw_datasets = datasets.load_from_disk(os.environ["FON_DATASET_PATH"])\n'
if source.count(start_marker) == 1 and source.count(end_marker) == 1:
    start = source.index(start_marker)
    end = source.index(end_marker, start)
    trainer_script.write_text(source[:start] + replacement + source[end:], encoding="utf-8")
elif (
    source.count(start_marker) == 0
    and source.count(local_dataset_marker) == 1
    and source.count(end_marker) == 1
    and 'raise TypeError("Le jeu local doit être un DatasetDict")' in source
):
    print("Chargement local déjà configuré dans ce runtime; aucune modification répétée.")
else:
    raise RuntimeError(
        "Structure du script amont inattendue : "
        f"bloc d'origine={source.count(start_marker)}, "
        f"bloc local={source.count(local_dataset_marker)}, "
        f"marqueur suivant={source.count(end_marker)}. "
        "Redémarre le runtime Colab et exécute les cellules dans l'ordre."
    )
os.environ["FON_DATASET_PATH"] = str(DATASET_PATH)
print(f"Fine-tuner vérifié et configuré pour le paquet local : {actual_commit}")

In [ ]:
CONVERTED_MODEL = Path("/content/mms_fon_trainable")
subprocess.run([
    sys.executable,
    "convert_original_discriminator_checkpoint.py",
    "--language_code", "fon",
    "--pytorch_dump_folder_path", str(CONVERTED_MODEL),
], cwd=REPO_DIR, check=True)
print(f"Checkpoint MMS Fon avec discriminateur prêt : {CONVERTED_MODEL}")

In [ ]:
from datetime import datetime
import json

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(f"/content/fon_tts_speaker4_pilot_{RUN_ID}")
if OUTPUT_DIR.exists():
    raise FileExistsError(f"Le dossier de sortie de cet essai existe déjà : {OUTPUT_DIR}")
config = {
    "project_name": "fon_speaker4_pilot",
    "push_to_hub": False,
    "overwrite_output_dir": False,
    "output_dir": str(OUTPUT_DIR),
    "model_name_or_path": str(CONVERTED_MODEL),
    "dataset_name": "local-fon-dataset",
    "audio_column_name": "audio",
    "text_column_name": "text",
    "train_split_name": "train",
    "eval_split_name": "eval",
    "max_duration_in_seconds": 20.0,
    "min_duration_in_seconds": 0.1,
    "max_tokens_length": 450,
    "max_train_samples": 64,
    "max_eval_samples": 16,
    "preprocessing_num_workers": 2,
    "do_train": True,
    "do_eval": True,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 1,
    "learning_rate": 1e-5,
    "adam_beta1": 0.8,
    "adam_beta2": 0.99,
    "warmup_ratio": 0.01,
    "weight_disc": 3,
    "weight_fmaps": 1,
    "weight_gen": 1,
    "weight_kl": 1.5,
    "weight_duration": 1,
    "weight_mel": 35,
    "evaluation_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 1,
    "logging_steps": 1,
    "dataloader_num_workers": 2,
    "fp16": True,
    "report_to": "none",
    "seed": 42,
}
CONFIG_PATH = Path("/content/fon_speaker4_pilot.json")
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")
command = [
    "accelerate", "launch", "--num_processes", "1",
    str(REPO_DIR / "run_vits_finetuning.py"), str(CONFIG_PATH),
]
print(f"Python Colab : {sys.version}")
result = subprocess.run(command, cwd=REPO_DIR, text=True, capture_output=True)
if result.stdout:
    print(result.stdout, end="")
if result.stderr:
    print(result.stderr, file=sys.stderr, end="")
if result.returncode != 0:
    raise RuntimeError(f"L'entraînement a échoué (code {result.returncode}). Consulte les logs complets juste au-dessus.")
if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError("Le script n'a pas produit le dossier du modèle adapté.")
print(f"Pilote terminé : {OUTPUT_DIR}")

In [ ]:
import json
import shutil
import soundfile as sf
from IPython.display import Audio, display
from transformers import VitsModel, set_seed

COMPARISON_DIR = OUTPUT_DIR / "comparison"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
comparison = json.loads((BUNDLE_ROOT / "comparison/comparison.json").read_text(encoding="utf-8"))
adapted_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
adapted_model = VitsModel.from_pretrained(OUTPUT_DIR).to("cuda").eval()
sampling_rate = adapted_model.config.sampling_rate

for index, item in enumerate(comparison, start=1):
    seed = 42 + index
    set_seed(seed)
    inputs = adapted_tokenizer(item["text"], return_tensors="pt").to("cuda")
    with torch.inference_mode():
        waveform = adapted_model(**inputs).waveform[0].detach().cpu().numpy()
    adapted_path = COMPARISON_DIR / f"{index:03d}_adapte.wav"
    sf.write(adapted_path, waveform, sampling_rate, subtype="PCM_16")
    reference_path = BUNDLE_ROOT / item["reference_audio"]
    original_path = BUNDLE_ROOT / item["original_model_audio"]
    print(f"Phrase {index}: {item['text']}")
    print("Voix réelle (validation)")
    display(Audio(filename=str(reference_path)))
    print("MMS original")
    display(Audio(filename=str(original_path)))
    print("MMS adapté — pilote")
    display(Audio(filename=str(adapted_path)))

adapted_model.save_pretrained(OUTPUT_DIR / "inference_model")
adapted_tokenizer.save_pretrained(OUTPUT_DIR / "inference_model")
DRIVE_RUN = DRIVE_ROOT / "runs" / f"speaker4_pilot_{RUN_ID}"
DRIVE_RUN.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(OUTPUT_DIR, DRIVE_RUN / "model")
shutil.copy2(BUNDLE_ROOT / "quality_report.json", DRIVE_RUN / "quality_report.json")
shutil.copy2(BUNDLE_ROOT / "comparison/comparison.json", DRIVE_RUN / "comparison.json")
print(f"Modèle et audios de comparaison sauvegardés dans : {DRIVE_RUN}")

## Interpréter le résultat

Écoute les trois versions de chaque phrase : enregistrement réel de validation, MMS original, puis pilote adapté. Le petit essai sert d'abord à confirmer que l'entraînement et l'export fonctionnent; il ne constitue pas une preuve de qualité ni de ressemblance vocale. Fais évaluer la prononciation et le naturel par des locuteurs du fon avant d'augmenter la durée d'entraînement. Garde le modèle adapté et les audios privés, et supprime-les de Drive lorsque tu n'en as plus besoin.